# 02 - Materials, tallies and run settings

**~30 minutes.**

In [ ]:
# Run this cell first. It sets up the paths used by the whole notebook.
from pathlib import Path

DATA = Path("..") / "data"        # the MCNP files, read-only
OUT = Path("..") / "outputs"      # everything we produce goes here
OUT.mkdir(exist_ok=True)

%matplotlib inline
print("data folder :", DATA.resolve())
print("output folder:", OUT.resolve())

In [ ]:
from copy import deepcopy

from f4enix.input.libmanager import LibManager
from f4enix.input.materials import Material
from f4enix.input.MCNPinput import Input

inp = Input.from_input(DATA / "iter_1d" / "ITER_1D.i")

## 1. The material section

`inp.mat_section` is the list of all `M` cards. You can index it by name. In the case of materials, migjorn parsing is improved and materials are handled directly by f4enix. This means that the materials are deleted from the migjorn ``._model`` object in order to avoid confusion.

In [ ]:
print([m.name for m in inp.mat_section])

The materials format is standardized by f4enix. One zaid per line and every zaid gets a dollar comment specifying:
- its formula.
- the weight percentage of the element (and not of the zaid) in the material definition.
- the abundance of the zaid in the element.

Multiple instances of the same zaid are collapsed into one line (i.e., their fractions are summed together).


In [ ]:
# M7 is the coolant. to_text() gives you the card as MCNP would see it.
print(inp.mat_section["M7"].to_text())

A `Material` is a structure, not a block of text. It is made of `Element`s, each made of
`Zaid`s.

In [ ]:
water = inp.mat_section["M7"]

for element in water.elements:
    print(f"element {element.name} (Z={element.Z})")
    for zaid in element.zaids:
        print(f"    {zaid.name:12s} fraction {zaid.fraction: .3e}")

In [ ]:
# some useful attributes of the Zaid object

print(f'Element number: {zaid.element}')
print(f'Isotope: {zaid.isotope}')
print(f'MCNP string: {zaid.name}')
print(f'Library: {zaid.library}')
print(f'Chemical formula: {zaid.fullname}')


`get_info()` turns all of that into a table, and adds the quantities you usually have to
compute by hand: atom fraction, mass fraction, and the same per element.

In [ ]:
inp.mat_section["M7"].get_info()

The same method on the whole section gives you every material at once - handy to search for a
specific nuclide across a model.

In [ ]:
info = inp.mat_section.get_info()
print("rows:", len(info))

# where is boron used in this model?
info[info.index.get_level_values("Element") == "B"].head(10)

## 2. The `LibManager`

Anything involving nuclear data - which libraries exist, which nuclides they contain, atomic
masses, natural abundances - goes through a `LibManager`.

You can point it at your own `xsdir` (in case you need specific libraries that you added to your xsdir file), but with no arguments it uses the data bundled with
F4Enix, which is what we do here.

In [ ]:
libman = LibManager()

print("default library    :", libman.defaultlib)
print("is 31c available?  :", libman.is_lib_available("31c"))
print("is 81c available?  :", libman.is_lib_available("81c"))

In [ ]:
# which libraries contain Fe-56?
print(sorted(str(lib) for lib in libman.check4zaid("26056")))

In [ ]:
# natural composition of an element, straight from the bundled data
libman.expand_element("Fe")

## 3. Building a new material

`Material.from_zaids` takes a list of `(zaid, fraction)` pairs. Positive fractions are atom
fractions, negative ones are mass fractions.

Using a **natural** zaid (`26000`, `8000`, ...) is the convenient trick: the translation step
expands it into its isotopes using the natural abundances.

In [ ]:
steel = Material.from_zaids(
    [(26000, -0.70), (24000, -0.18), (28000, -0.12)],   # mass fractions
    libman,
    "31c",
    name="A rough austenitic steel",
    mat_id=95,
)

print(steel.to_text())

materials can be created also from the element names

In [ ]:
mat = Material.from_zaids(
    [("H2", 2), ("O", 1)],
    libman,
    '31c',
    name="Heavy Water")

print(mat.to_text())

Adding it to the model is a plain list append.

In [ ]:
inp.mat_section.append(steel)
inp.mat_section['M95'].header

## 4. Mixing materials that already exist

`generate_material` builds a homogenised material out of materials already in the section -
the everyday operation when you smear a component. It returns the new material, so you decide
whether to keep it.

In [ ]:
mixed = inp.mat_section.generate_material(
    ["M7", "M11"],          # water and steel
    [0.4, 0.6],             # 40% / 60%
    "31c",
    libman,
    fractiontype="atom",
    mat_name="M96",
)

print(mixed.to_text()[:500])

## 5. Changing cross-section library

`translate` rewrites the library suffix of every zaid. It can be called on a single material or
on the whole input.

Watch the last line of the result carefully.

In [ ]:
libman.defaultlib = '31c'
m7_81c = deepcopy(inp.mat_section["M7"])
m7_81c.translate("81c", libman)

print(m7_81c.to_text())

`8018` came out with the 31c suffix. That is not a bug: O-18 simply does not exist in `81c`, so F4Enix falls back to its default ('31c' in this case). This is exactly
the kind of thing you want to catch *before* submitting a run - and the reason translation is
worth doing with a tool rather than with find-and-replace.

In [ ]:
# the same call on the whole model translates every material at once
translated = Input.from_input(DATA / "iter_1d" / "ITER_1D.i")
translated.translate("81c", libman)

print(translated.mat_section["M11"].to_text()[:300])

`translate` also accepts a dictionary, if you only want to move some libraries:

```python
inp.translate({"31c": "81c"}, libman)          # only what is currently 31c
inp.translate({"81c": ["1001", "8016"]}, libman)  # only these zaids
```

## 6. Fractions and densities

Two conversions come up constantly.

**Atom fractions <-> mass fractions.** `switch_fraction` with `inplace=False` returns a copy, so
the original material is untouched.

In [ ]:
as_mass = inp.mat_section["M7"].switch_fraction("mass", libman, inplace=False)
print(as_mass.to_text())

Which methods modify the material and which do not? The cell below prints `M7` three times:

1. after `get_tad` - nothing changed: `get_tad` works on an internal copy, the fractions are still atom fractions (positive);
2. after `switch_fraction("mass", ...)` - this time the material **did** change: `inplace=True` is the default, so the fractions are now mass fractions (negative);
3. after `get_tad` again - still mass fractions: `get_tad` never touches the material, whatever its fraction type.

Keep in mind that, from here on, `M7` in this notebook is written in mass fractions.


In [ ]:
inp.mat_section["M7"].get_tad(1.0, libman)
print(inp.mat_section["M7"].to_text())
inp.mat_section["M7"].switch_fraction('mass', libman)
print(inp.mat_section["M7"].to_text())
inp.mat_section["M7"].get_tad(1.0, libman)
print(inp.mat_section["M7"].to_text())

**Mass density <-> total atom density.** these are often needed when preparing tallies.

In [ ]:
m7 = deepcopy(inp.mat_section["M7"])   # get_density modifies the material, see below

print("1.0 g/cc of M7   ->", m7.get_tad(1.0, libman), "atoms/barn/cm")
print("1.0028e-1 at/b/cm ->", m7.get_density(1.0028e-1, libman), "g/cc")

> **Careful:** `get_density` modifies the material it is called on (it switches its fractions to
> mass). Call it on a `deepcopy` if you still need the original, as we did above.

## 7. Changing what is in the cells

`scale_densities` multiplies every non-void cell density by a factor - a one-liner for a
sensitivity study.

In [ ]:
scaled = Input.from_input(DATA / "iter_1d" / "ITER_1D.i")

print("before:", scaled.cells["34"].density)
scaled.scale_densities(0.9)
print("after :", scaled.cells["34"].density)

`replace_material` swaps a material for another one in every cell that uses it, and sets a new
density at the same time. An optional `u_list` restricts the change to given universes.

In [ ]:
swapped = Input.from_input(DATA / "iter_1d" / "ITER_1D.i")

swapped.replace_material(95, -7.9, 11)          # material 11 -> 95, at 7.9 g/cc
print(swapped.cells["33"])
print(swapped.get_cells_summary()["material"].value_counts().sort_index())

## 8. Tallies

`get_tally_summary` is the fastest way to see what a model is actually scoring. The
`Normalization` column is the `FM` multiplier, which is where the units come from. This can be especially useful when generating reports on your activities.

In [ ]:
inp.get_tally_summary()

# remember that pd.DataFrames can be easily exported to CSV or excel
# with .to_excel() and .to_csv()

In [ ]:
# the FMESH tallies are listed separately
inp.get_tally_summary(fmesh=True)

`add_F_tally` writes the whole set of cards for you: `FC` (the comment), `F`, `E` (the energy
grid) and `SD` (the segment divisor).

In [ ]:
inp.add_F_tally(
    304,
    ["n"],
    [86, 87, 88],
    energies=[1e-6, 0.1, 1.0, 14.0],
    description="A test tally in the outboard magnet",
    add_total=True,
)

for key in ("FC304", "F304", "E304", "SD304"):
    print(inp.other_data[key])

In [ ]:
print("tallies now:", inp.tally_keys)

`remove_tallies` takes a list of IDs, or nothing at all to remove every tally in the model.

In [ ]:
inp.remove_tallies([304])
print("tallies now:", inp.tally_keys)

## 9. Run settings

`add_stopCard` sets `NPS`. Despite the name it does not add a `STOP` card any more.

In [ ]:
inp.add_stopCard(5e7)
print(inp.other_data["NPS"])

`prepare_void_check` builds a complete void-check run in one call: it removes the source and
every tally, adds `VOID`, and creates a surface source on a **spherical** boundary of the model.

`ITER_1D` is bounded by cylinders, so we use the small sphere model to demonstrate it.

In [ ]:
sphere = Input.from_input(DATA / "sphere" / "Sphere.i")
sphere.prepare_void_check(3, 1_000_000)      # surface 3 is the outer sphere

for key in ("VOID", "NPS", "SDEF", "SI1", "SP1"):
    print(sphere.other_data[key])

---

# Exercises

## Exercise 1 - swap the coolant

Replace the coolant of the model (material 7) with a new material:

1. build material **99** from **natural** hydrogen and oxygen (zaids `1000` and `8000`) with
   atom fractions 2 and 1, on library `31c`, and give it a name;
2. add it to the material section;
3. make every cell that used material 7 use material 99 instead, at a mass density of
   **1.0 g/cc**.

<details>
<summary><b>Hints</b> <- Click to expand</summary>

* `Material.from_zaids(zaids, libman, lib, name=..., mat_id=...)`, where `zaids` is a list of
  `(zaid, fraction)` pairs;
* `inp.mat_section.append(...)`;
* `inp.replace_material(new_id, new_density, old_id)` - in MCNP a **mass** density is negative.

</details>

In [ ]:
inp = Input.from_input(DATA / "iter_1d" / "ITER_1D.i")

# TODO 1: build the new water material, id 99, library 31c, from natural H and O
#         (natural zaids are 1000 and 8000, atom fractions 2 and 1)
water = ...

# TODO 2: add it to the material section of the model


# TODO 3: replace material 7 by material 99 everywhere, at a mass density of 1.0 g/cc
#         (remember: mass densities are negative in MCNP)


print(water.to_text())

In [ ]:
# --- self-check, do not modify ---
names = [m.name for m in inp.mat_section]
assert "M99" in names, f"M99 was not added, materials are {names}"

m99 = inp.mat_section["M99"]
assert len(m99.zaids) == 5, (
    f"M99 has only {len(m99.zaids)} zaids - did you use the natural zaids 1000 and 8000?"
)

counts = inp.get_cells_summary()["material"].value_counts()
assert 7 not in counts.index, "there are still cells using material 7"
assert counts.get(99, 0) == 14, f"expected 14 cells with material 99, found {counts.get(99, 0)}"
assert float(inp.cells["34"].density) == -1.0, f"cell 34 density is {inp.cells['34'].density}"
print("Exercise 1 correct")

## Exercise 2 - a new tally and a run size

Still on the same `inp` object, add a neutron flux tally on the **outboard magnet**, cells
86 to 105:

1. tally number **400**, particle `n`, on those cells, with a **total** bin;
2. energy grid `[1e-6, 0.1, 1.0, 5.0, 14.0]`;
3. a description of your choice;
4. set the number of histories to **5e7**;
5. write the model to `outputs/iter_1d_variant.i`.


<details>
<summary><b>Hints</b> <- Click to expand</summary>

* `add_F_tally(id, particles, cells, add_total=..., energies=..., description=...)` -
  `particles` is a **list**, e.g. `["n"]`;
* `add_stopCard(5e7)`;
* `inp.write(OUT / "iter_1d_variant.i")`.

</details>

In [ ]:
# TODO 1: the cells of the outboard magnet, 86 to 105
magnet_cells = ...

# TODO 2: add tally 400, neutrons, on those cells, with the energy grid below,
#         a description, and a total bin
energy_grid = [1e-6, 0.1, 1.0, 5.0, 14.0]


# TODO 3: set NPS to 5e7


# TODO 4: write the model to OUT / "iter_1d_variant.i"


print(inp.other_data["F400"])

In [ ]:
# --- self-check, do not modify ---
reloaded = Input.from_input(OUT / "iter_1d_variant.i")
keys = {k.upper(): str(v) for k, v in reloaded.other_data.items()}
f400 = "".join(v for k, v in keys.items() if k.startswith("F400"))

assert 400 in reloaded.tally_keys, f"tally 400 is missing, found {reloaded.tally_keys}"
assert "E400" in keys, "the energy grid card E400 is missing"
assert "1.4000e+01" in keys["E400"], "the energy grid does not reach 14 MeV"
assert "105" in f400, "cell 105 is not in the tally"
assert "50000000" in keys["NPS"], f"NPS is {keys['NPS']}"
print("Exercise 2 correct")

---

## What you can now do

| Question | Method |
|----------|--------|
| what is in material X? | `mat_section["M7"].get_info()`, `.to_text()` |
| which libraries have this nuclide? | `LibManager.check4zaid` |
| what is the natural composition of Fe? | `LibManager.expand_element` |
| build a material from scratch | `Material.from_zaids` |
| smear two materials together | `mat_section.generate_material` |
| move the model to another library | `translate` |
| atom fraction <-> mass fraction | `switch_fraction` |
| mass density <-> atom density | `get_tad`, `get_density` |
| swap a material everywhere | `replace_material` |
| add / remove a tally | `add_F_tally`, `remove_tallies` |
| set the run size | `add_stopCard` |
| set up a void check | `prepare_void_check` |

The input is ready. In the next notebook the run has been done and we look at what came out of
it: **`03_parsing_outp_and_mctal.ipynb`**.